<a href="https://colab.research.google.com/github/klizaola/beginning-bioinformatics/blob/main/Lizaola_Gut_to_Soil_QIIME2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gut-to-soil microbiome analysis

Name: **[Kayla Lizaola]** &nbsp; Date: **[10/29/26]**

Follow the [Gut-to-soil tutorial, release 2026.10](https://gut-to-soil-tutorial.readthedocs.io/en/2026.10/). Use its **Command Line** tabs.

Your goal is to turn paired-end 16S reads into a community profile, compare environments, and connect the analysis to machine learning.

### How to use this notebook
1. Run the three **Provided setup** sections unchanged.
2. Work through the tutorial and notebook together, one cell at a time. Do **not** use Run all while placeholders remain.
3. For each numbered task, copy the indicated `qiime ...` command from the tutorial's **Command Line** tab and paste it **between** the triple quotes, replacing `PASTE_TUTORIAL_COMMAND_HERE`. Include its option lines and backslashes. Do not paste Markdown fences, `$`, `!`, `cd`, or the Python API example.
4. Keep tutorial filenames. The provided `tutorial_step` function runs the command in the installed QIIME environment and correct directory. You do not need to activate an environment or add `!`.
5. Use the visualization selector after each section to download `.qzv` outputs; open them in [QIIME 2 View](https://view.qiime2.org/). Write observations in the response cells.
6. Save your notebook with responses and download the results ZIP at the end.

One **"Worked example"** is included before the first task. Blank cells stop with a clear message rather than running an answer for you.


### Colab adjustments handled for you
The adapter prints the final command before running it. It limits DADA2 threads, taxonomy-training batch size, and taxonomy-classification workers/batches. It saves the classifier annotation to a separate file and uses that annotated copy downstream. Provenance paths are adjusted to this notebook's folder. All biological settings remain those in your pasted command. Reference artifacts and input reads are provided by download helpers so the exercise focuses on analytical commands.


## Provided setup A — Settings and file helpers

Keep the release fixed. The configuration records downloaded-file checksums and the installed environment. The release environment is pinned by its published specification; its URL is not a content-addressed lockfile. Downloads and generated results live on the Colab VM until you export them.

If you change an analysis parameter, rerun that cell and every downstream cell. The runner detects changed inputs/commands and recomputes outputs. The student tasks compute DADA2; the recovery section provides separately labeled reference plots if needed.


In [52]:
from pathlib import Path
import os, sys, json, shlex, shutil, subprocess, hashlib, time, urllib.request
import tarfile, zipfile, platform, html

RELEASE = "2026.10"
DADA2_MODE = "compute"             # "compute" or "checkpoint"
REFERENCE_SOURCE = "tutorial-artifacts"  # or "rebuild" (download from GTDB)
THREADS = 2
TRAINING_CHUNK_SIZE = 200          # tutorial uses 2000; smaller lowers transient RAM
SAMPLING_DEPTH = 96                # shallow tutorial subset, not a research recommendation
RAREFACTIONS = 10
SEED = 42
assert DADA2_MODE in {"compute", "checkpoint"}
assert REFERENCE_SOURCE in {"tutorial-artifacts", "rebuild"}
assert platform.system() == "Linux" and platform.machine() in {"x86_64", "AMD64"}

ROOT = Path('/content/gut_to_soil_student')
WORK = ROOT / 'analysis'
LOGS = ROOT / 'logs'
for p in (ROOT, WORK, LOGS): p.mkdir(parents=True, exist_ok=True)
MM = ROOT / 'bin' / 'micromamba'
PREFIX = ROOT / ('qiime-' + RELEASE)
BASE = f'https://gut-to-soil-tutorial.readthedocs.io/en/{RELEASE}/data/gut-to-soil/'
ENV_URL = f'https://raw.githubusercontent.com/qiime2/distributions/refs/heads/dev/{RELEASE}/qiime2/released/rachis-qiime2-linux-64-conda.yml'
RUN_ENV = os.environ.copy()
# Prevent notebook-side Python settings from leaking into the QIIME process.
for key in ('PYTHONPATH', 'PYTHONHOME'): RUN_ENV.pop(key, None)
RUN_ENV.update(MAMBA_ROOT_PREFIX=str(ROOT / 'mamba'), PYTHONNOUSERSITE='1',
               MPLBACKEND='Agg', OMP_NUM_THREADS=str(THREADS),
               OPENBLAS_NUM_THREADS='1', MKL_NUM_THREADS='1')

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda: f.read(1024*1024), b''): h.update(b)
    return h.hexdigest()

def record(name, item):
    p = ROOT / name
    data = json.loads(p.read_text()) if p.exists() else []
    data.append(dict(time=time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), **item))
    p.write_text(json.dumps(data, indent=2))

def download(url, dest, *, refresh=False):
    dest = Path(dest); dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size and not refresh:
        return dest
    temp = dest.with_name(dest.name + '.part')
    for attempt in range(3):
        try:
            req = urllib.request.Request(url, headers={'User-Agent':'GutToSoilTeachingNotebook/1.0'})
            with urllib.request.urlopen(req, timeout=120) as response, temp.open('wb') as out:
                shutil.copyfileobj(response, out)
            if not temp.stat().st_size: raise ValueError('Empty download')
            if dest.suffix in {'.qza', '.qzv', '.zip'}:
                if not zipfile.is_zipfile(temp): raise ValueError('Expected ZIP/QIIME artifact; got another file type')
                with zipfile.ZipFile(temp) as z:
                    bad = z.testzip()
                    if bad: raise ValueError('Corrupt ZIP member: '+bad)
            elif dest.suffix in {'.tsv', '.yml'}:
                beginning = temp.read_text()[:200].lower()
                if '<html' in beginning or '<!doctype' in beginning:
                    raise ValueError('Received an HTML page instead of data')
            temp.replace(dest)
            record('downloads.json', dict(url=url, path=str(dest.relative_to(ROOT)),
                   bytes=dest.stat().st_size, sha256=sha256(dest)))
            print('Downloaded', dest.name, f'({dest.stat().st_size/1e6:.1f} MB)')
            return dest
        except Exception:
            temp.unlink(missing_ok=True)
            if attempt == 2: raise
            time.sleep(2)

def fetch(name, dest=None):
    return download(BASE + name, WORK / (dest or name))

def run(argv, log_name='command', cwd=None):
    argv = [str(a) for a in argv]
    print('$', shlex.join(argv), flush=True)
    log = LOGS / (log_name + '.log')
    started = time.monotonic()
    with log.open('w') as f:
        p = subprocess.Popen(argv, cwd=cwd or WORK, env=RUN_ENV,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1)
        try:
            for line in p.stdout:
                print(line, end='', flush=True); f.write(line); f.flush()
            rc = p.wait()
        except BaseException:
            p.terminate()
            try: p.wait(timeout=10)
            except subprocess.TimeoutExpired: p.kill(); p.wait()
            raise
    record('runs.json', dict(command=argv, returncode=rc,
           seconds=round(time.monotonic()-started, 2), log=str(log.relative_to(ROOT))))
    if rc:
        raise RuntimeError(f'Command failed (exit {rc}). Read {log}. No checkpoint was substituted.')

def qcmd(args): return [MM, 'run', '-p', PREFIX, 'qiime', *args]

mem = dict(line.split(':',1) for line in Path('/proc/meminfo').read_text().splitlines())
print('Available RAM:', round(int(mem['MemAvailable'].split()[0])/1024**2,1), 'GiB')
print('Free disk:', round(shutil.disk_usage(ROOT).free/1024**3,1), 'GiB')
print('Notebook Python:', sys.executable, '(will remain unchanged)')


Available RAM: 11.2 GiB
Free disk: 216.9 GiB
Notebook Python: /usr/bin/python3 (will remain unchanged)


## Provided setup B — Installation

This uses the official release environment with micromamba as its Conda-compatible installer. It does not replace Colab's Python. Installation may be lengthy; inspect the live output. If it fails, keep the installation log rather than adding arbitrary `pip install` fixes. The initial free-space check is a conservative classroom guard, not an official minimum.


In [53]:
if not MM.exists():
    package = download('https://micro.mamba.pm/api/micromamba/linux-64/latest', ROOT/'micromamba.tar.bz2')
    with tarfile.open(package) as tar:
        member = tar.getmember('bin/micromamba')
        MM.parent.mkdir(exist_ok=True)
        with tar.extractfile(member) as src, MM.open('wb') as dst:
            shutil.copyfileobj(src, dst)
    MM.chmod(0o755)
run([MM, '--version'], 'micromamba-version')
env_file = download(ENV_URL, ROOT/f'qiime-{RELEASE}.yml')
ready = PREFIX / '.notebook-install-complete'
if not ready.exists():
    if shutil.disk_usage(ROOT).free < 20*1024**3:
        raise RuntimeError('Less than 20 GiB free. Start a fresh runtime or free space before installation.')
    run([MM, 'create', '-y', '-p', PREFIX, '-f', env_file], 'installation')
    run(qcmd(['info']), 'qiime-info')
    ready.write_text(sha256(env_file))
else:
    assert ready.read_text() == sha256(env_file), 'Environment specification changed; use a fresh runtime.'
    run(qcmd(['info']), 'qiime-info')
run([MM, 'list', '-p', PREFIX, '--explicit'], 'environment-explicit')
print('Installation passed qiime info. Next: test the actual tutorial actions.')


$ /content/gut_to_soil_student/bin/micromamba --version
2.10.0
$ /content/gut_to_soil_student/bin/micromamba run -p /content/gut_to_soil_student/qiime-2026.10 qiime info
System versions
Python version: 3.12.15
Parsl version: 2026.2.23
rachis release: 2026.10
rachis version: 2026.10.0
rachis-cli version: 2026.10.0

Installed plugins
alignment: 2026.10.0
boots: 2026.10.0
composition: 2026.10.0
cutadapt: 2026.10.0
dada2: 2026.10.0
deblur: 2026.10.0
demux: 2026.10.0
diversity: 2026.10.0
diversity-lib: 2026.10.0
emperor: 2026.10.0
fastp: 2026.10.0
feature-classifier: 2026.10.0
feature-table: 2026.10.0
fondue: 2026.10.0
fragment-insertion: 2026.10.0
kmerizer: 2026.10.0
longitudinal: 2026.10.0
metadata: 2026.10.0
phylogeny: 2026.10.0
quality-control: 2026.10.0
quality-filter: 2026.10.0
rescript: 2026.10.0
sample-classifier: 2026.10.0
stats: 2026.10.0
taxa: 2026.10.0
types: 2026.10.0
vizard: 2026.10.0
vsearch: 2026.10.0

Application config directory
/content/gut_to_soil_student/qiime-2026.10/v

## Provided setup C — Compatibility checks and runner

This checks release-specific command flags before the analysis starts. A successful preflight verifies the CLI, not the analysis itself. Each step writes a success record only after all expected outputs exist. Changed input bytes or changed commands invalidate cached results. Partial outputs are regenerated on retry.


In [58]:
checks = {
 ('dada2','denoise-paired'): ['--o-base-transition-stats','--p-n-threads'],
 ('feature-table','summarize'): ['--o-summary','--o-sample-frequencies','--m-metadata-file'],
 ('feature-table','tabulate-seqs'): ['--i-taxonomy'],
 ('feature-classifier','fit-classifier-naive-bayes'): ['--p-classify--chunk-size'],
 ('feature-classifier','classify-sklearn'): ['--p-reads-per-batch','--p-n-jobs'],
 ('boots','kmer-diversity'): ['--p-n','--p-no-replacement'],
 ('composition','ancombc2'): ['--p-fixed-effects-formula','--o-ancombc2-output'],
 ('composition','da-barplot'): ['--i-data'],
 ('rescript','get-gtdb-data'): ['--p-db-type'],
 ('tools','annotation-create'): ['--annotation-type'],
 ('tools','replay-provenance'): ['--in-fp','--out-fp'],
}
problems=[]
for action, flags in checks.items():
    p = subprocess.run([str(x) for x in qcmd([*action,'--help'])], env=RUN_ENV,
                       text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    (LOGS/('help-'+'-'.join(action)+'.log')).write_text(p.stdout)
    missing=[f for f in flags if f not in p.stdout]
    if p.returncode or missing: problems.append((' '.join(action),missing,p.returncode))
if problems: raise RuntimeError(f'Release/API mismatch: {problems}. See help logs; do not mix tutorial releases.')
print('All tutorial API checks passed.')

def digest_path(p):
    p=Path(p)
    if p.is_file(): return sha256(p)
    if p.is_dir():
        return hashlib.sha256(json.dumps([(str(f.relative_to(p)),sha256(f))
             for f in sorted(p.rglob('*')) if f.is_file()]).encode()).hexdigest()
    return None

def step(name, command, outputs):
    argv=shlex.split(command)
    assert argv[0]=='qiime'
    paths=[WORK/o for o in outputs]
    for p in paths:
        assert p.resolve().is_relative_to(WORK.resolve()) and p.resolve()!=WORK.resolve()
    inputs={}
    for token in argv[1:]:
        candidate=WORK/token
        # Named collection arguments use label:path.
        if not candidate.exists() and ':' in token: candidate=WORK/token.split(':',1)[1]
        if candidate.exists() and candidate not in paths:
            inputs[str(candidate.relative_to(WORK))]=digest_path(candidate)
    signature=hashlib.sha256(json.dumps([command,inputs,RELEASE],sort_keys=True).encode()).hexdigest()
    marker=LOGS/('step-'+hashlib.sha256(name.encode()).hexdigest()[:12]+'.json')
    old=json.loads(marker.read_text()) if marker.exists() else {}
    observed={o:digest_path(WORK/o) for o in outputs}
    if old.get('signature')==signature and all(observed.values()) and old.get('outputs')==observed:
        print('Already completed:',name); return
    marker.unlink(missing_ok=True)
    for p in paths:
        if p.is_dir(): shutil.rmtree(p)
        elif p.exists(): p.unlink()
    run(qcmd(argv[1:]), 'step-'+hashlib.sha256(name.encode()).hexdigest()[:12])
    observed={o:digest_path(WORK/o) for o in outputs}
    if not all(observed.values()): raise RuntimeError(f'Missing outputs for {name}: {observed}')
    marker.write_text(json.dumps(dict(name=name,signature=signature,outputs=observed),indent=2))

def view_outputs():
    from IPython.display import display, HTML
    import ipywidgets as w
    paths=sorted(WORK.rglob('*.qzv'))
    if not paths: print('No visualizations yet.'); return
    choices=w.Dropdown(options=[(str(p.relative_to(WORK)),str(p)) for p in paths],
                       description='Result:',layout=w.Layout(width='90%'))
    button=w.Button(description='Download selected .qzv')
    def selected(_):
        from google.colab import files
        files.download(choices.value)
    button.on_click(selected)
    display(HTML('<p>Download a visualization, then open it at <a href="https://view.qiime2.org/" target="_blank">QIIME 2 View</a>. Inspect its provenance as well as its plots.</p>'))
    display(choices,button)


All tutorial API checks passed.


## Provided setup D — Paste-in adapter
Run once. This accepts a **single QIIME command** per task. It recognizes the requested outputs, applies the Colab adjustments listed above, and passes the command to the tested runner. It does not fill in missing commands or invent results.


In [59]:
def tutorial_step(task, command):
    if 'PASTE_TUTORIAL_COMMAND_HERE' in command or not command.strip():
        raise ValueError(f'{task}: paste the indicated tutorial command between the triple quotes first.')
    command = command.replace('\\\n', '')
    args = shlex.split(command)
    if not args or args[0] != 'qiime' or args.count('qiime') != 1:
        raise ValueError('Paste exactly one command beginning with qiime, without !, $, cd, or code fences.')
    if any(x in args for x in [';', '&&', '|']):
        raise ValueError('Paste one command only, without shell operators.')
    def option(flag, value):
        if flag in args:
            j=args.index(flag)
            if j+1==len(args): raise ValueError(f'Missing value after {flag}')
            args[j+1]=str(value)
        else: args.extend([flag,str(value)])
    action=tuple(args[1:3])
    if action == ('dada2','denoise-paired'): option('--p-n-threads', THREADS)
    if action == ('feature-classifier','fit-classifier-naive-bayes'):
        option('--p-classify--chunk-size',TRAINING_CHUNK_SIZE)
    if action == ('feature-classifier','classify-sklearn'):
        option('--p-n-jobs',1); option('--p-reads-per-batch',100)
        j=args.index('--i-classifier')+1
        if args[j]=='suboptimal-16S-rRNA-classifier.qza': args[j]='teaching-classifier.qza'
    if action == ('tools','annotation-create'):
        option('--output-path','teaching-classifier.qza')
    if action == ('tools','replay-provenance'):
        option('--in-fp',WORK); option('--out-fp',ROOT/'g2s-replayed.bash')
        run(qcmd(args[1:]),'student-provenance')
        return
    outputs=[]
    for i,arg in enumerate(args):
        if arg.startswith('--o-') or arg in {'--output-dir','--output-path'}:
            if i+1==len(args): raise ValueError(f'Missing filename after {arg}')
            outputs.append(args[i+1])
    if not outputs: raise ValueError('No output flags found. Paste the entire command, including output options.')
    print('Command to run:',shlex.join(args))
    step(task,shlex.join(args),outputs)


## Worked example — Metadata
The tutorial's **Reviewing sample metadata and running a QIIME 2 command** section creates a metadata visualization. Notice that `--m-input-file` identifies metadata and `--o-visualization` names an output. Everything between the triple quotes is a normal tutorial command.

The downloads below are provided. Metadata describes each sample; `demux.qza` contains the already imported paired-end reads.


In [60]:
fetch('sample-metadata.tsv')
fetch('demux.qza')


PosixPath('/content/gut_to_soil_student/analysis/demux.qza')

In [61]:
tutorial_step("Worked example: metadata", r"""
qiime metadata tabulate \
  --m-input-file sample-metadata.tsv \
  --o-visualization sample-metadata.qzv
""")
view_outputs()


Command to run: qiime metadata tabulate --m-input-file sample-metadata.tsv --o-visualization sample-metadata.qzv
Already completed: Worked example: metadata


Dropdown(description='Result:', layout=Layout(width='90%'), options=(('alpha-rarefaction.qzv', '/content/gut_t…

Button(description='Download selected .qzv', style=ButtonStyle())

## 4. Get the tutorial reads and metadata

**Find in the tutorial:** Summarize demultiplexed sequences.

Before executing, identify what each input contains and predict what this step will produce.


### Task 1: Read quality

Find the command beginning **`qiime demux summarize`**.

**Expected output(s):** `demux.qzv`.


In [62]:
tutorial_step('Task 1: Read quality', r"""
qiime demux summarize \
  --i-data demux.qza \
  --o-visualization demux.qzv""")


Command to run: qiime demux summarize --i-data demux.qza --o-visualization demux.qzv
Already completed: Task 1: Read quality


In [63]:
view_outputs()


Dropdown(description='Result:', layout=Layout(width='90%'), options=(('alpha-rarefaction.qzv', '/content/gut_t…

Button(description='Download selected .qzv', style=ButtonStyle())

**Pause and explain:** What are the observations in the metadata? Which column describes sample environment? Record one observation from the read-quality plot.

**Your response:**

Observations in the metadata include sample color, moisture, contents, smell, and organisms (presence or absence, y/n). The sample type column describes the sample environment. One observation from the read-quality plot: the forward read for position 102 received a quality score of 14.


## 5. DADA2: reads → ASVs and counts

**Find in the tutorial:** Sequence quality control and feature table construction.

Before executing, identify what each input contains and predict what this step will produce.

**Prediction:** Before running, I predict that this step will trim and perform QC steps on the sequences. The inputs include the demultiplexed sequences, trimming and truncating the forward and referse strands, and denoising the data.


### Task 2: DADA2

Find the command beginning **`qiime dada2 denoise-paired`**.

**Expected output(s):** `asv-seqs.qza`, `asv-table.qza`, `denoising-stats.qza`, `base-transition-stats.qza`.


In [9]:
tutorial_step('Task 2: DADA2', r"""
qiime dada2 denoise-paired \
  --i-demultiplexed-seqs demux.qza \
  --p-trim-left-f 0 \
  --p-trunc-len-f 250 \
  --p-trim-left-r 0 \
  --p-trunc-len-r 250 \
  --o-representative-sequences asv-seqs.qza \
  --o-table asv-table.qza \
  --o-denoising-stats denoising-stats.qza \
  --o-base-transition-stats base-transition-stats.qza
""")


Command to run: qiime dada2 denoise-paired --i-demultiplexed-seqs demux.qza --p-trim-left-f 0 --p-trunc-len-f 250 --p-trim-left-r 0 --p-trunc-len-r 250 --o-representative-sequences asv-seqs.qza --o-table asv-table.qza --o-denoising-stats denoising-stats.qza --o-base-transition-stats base-transition-stats.qza --p-n-threads 2
$ /content/gut_to_soil_student/bin/micromamba run -p /content/gut_to_soil_student/qiime-2026.10 qiime dada2 denoise-paired --i-demultiplexed-seqs demux.qza --p-trim-left-f 0 --p-trunc-len-f 250 --p-trim-left-r 0 --p-trunc-len-r 250 --o-representative-sequences asv-seqs.qza --o-table asv-table.qza --o-denoising-stats denoising-stats.qza --o-base-transition-stats base-transition-stats.qza --p-n-threads 2
Saved FeatureTable[Frequency] to: asv-table.qza
Saved FeatureData[Sequence] to: asv-seqs.qza
Saved SampleData[DADA2Stats] to: denoising-stats.qza
Saved DADA2BaseTransitionStats to: base-transition-stats.qza


### Task 3: DADA2 statistics

Find the command beginning **`qiime metadata tabulate`**.

**Expected output(s):** `denoising-stats.qzv`.


In [64]:
tutorial_step('Task 3: DADA2 statistics', r"""
qiime metadata tabulate \
  --m-input-file denoising-stats.qza \
  --o-visualization denoising-stats.qzv
""")


Command to run: qiime metadata tabulate --m-input-file denoising-stats.qza --o-visualization denoising-stats.qzv
Already completed: Task 3: DADA2 statistics


### Task 4: Combined metadata

Find the command beginning **`qiime metadata tabulate`**.

**Expected output(s):** `sample-metadata-w-dada2-stats.qzv`.


In [65]:
tutorial_step('Task 4: Combined metadata', r"""
qiime metadata tabulate \
  --m-input-file sample-metadata.tsv denoising-stats.qza \
  --o-visualization sample-metadata-w-dada2-stats.qzv
""")


Command to run: qiime metadata tabulate --m-input-file sample-metadata.tsv denoising-stats.qza --o-visualization sample-metadata-w-dada2-stats.qzv
Already completed: Task 4: Combined metadata


In [66]:
view_outputs()

Dropdown(description='Result:', layout=Layout(width='90%'), options=(('alpha-rarefaction.qzv', '/content/gut_t…

Button(description='Download selected .qzv', style=ButtonStyle())

**Pause and explain:** Which processing stage lost the most reads for one sample you chose? Explain the distinction between an ASV, a read, and a species.

**Your response:**

The denoise stage lost the most reads for sample bcc246ab. It went from an initial 47 reads input, filtered to 43 reads, and then resulted in 3 reads after denoising. ASV stands for amplicon sequencing variant, which is a highly refined sequence of nucleotides (exact). A read is a single strand of DNA or RNA (ex. AGGTAGCTGGA) associated to a given sample. These result from processing the raw reads from the sample. Each sample can have multiple reads of short DNA or RNA sequences that are then cleaned up (filter/denoise) and used to infer nucleotide-exact ASVs. In the example given above, bcc246ab started with 47 reads and was filtered and denoised down to 3 reads. A species is a group of organisms from which the samples were taken whose ASVs maintain a certain degree of similarity. Since ASVs are very fine resolution and even single nucleotide differences can be detected, the definition of a species (whether you would call two samples with a single nucleotide difference between them different species) is up to the researcher and the question they are asking.

## 6. Summarize the count table

**Find in the tutorial:** Feature table and feature data summaries.

Before executing, identify what each input contains and predict what this step will produce.


### Task 5: 6. Summarize the count table

Find the command beginning **`qiime feature-table summarize`**.

**Expected output(s):** `asv-table.qzv`, `sample-frequencies.qza`, `asv-frequencies.qza`.


In [68]:
tutorial_step('Task 5: 6. Summarize the count table', r"""
qiime feature-table summarize \
  --i-table asv-table.qza \
  --m-metadata-file sample-metadata.tsv \
  --o-summary asv-table.qzv \
  --o-sample-frequencies sample-frequencies.qza \
  --o-feature-frequencies asv-frequencies.qza
""")


Command to run: qiime feature-table summarize --i-table asv-table.qza --m-metadata-file sample-metadata.tsv --o-summary asv-table.qzv --o-sample-frequencies sample-frequencies.qza --o-feature-frequencies asv-frequencies.qza
Already completed: Task 5: 6. Summarize the count table


### Task 6: ASV sequences

Find the command beginning **`qiime feature-table tabulate-seqs`**.

**Expected output(s):** `asv-seqs.qzv`.


In [69]:
tutorial_step('Task 6: ASV sequences', r"""
qiime feature-table tabulate-seqs \
  --i-data asv-seqs.qza \
  --m-metadata-file asv-frequencies.qza \
  --o-visualization asv-seqs.qzv
""")


Command to run: qiime feature-table tabulate-seqs --i-data asv-seqs.qza --m-metadata-file asv-frequencies.qza --o-visualization asv-seqs.qzv
Already completed: Task 6: ASV sequences


In [70]:
view_outputs()


Dropdown(description='Result:', layout=Layout(width='90%'), options=(('alpha-rarefaction.qzv', '/content/gut_t…

Button(description='Download selected .qzv', style=ButtonStyle())

**Pause and explain:** How many samples and ASVs remain? Find the most frequent ASV and record a sequence-based observation about it.

**Your response:**

1,069 samples and ASVs remain. The most frequent ASV is c6c3ab4e828fb40d6e05967b7aac9338. A sequence based observation about this ASV is that it contains a string of 5 Gs (GGGGG) in the first 15 nucleotides.

## 7. Filter rare features

**Find in the tutorial:** Filtering features from a feature table.

Before executing, identify what each input contains and predict what this step will produce.


### Task 7: 7. Filter rare features

Find the command beginning **`qiime feature-table filter-features`**.

**Expected output(s):** `asv-table-ms2.qza`.


In [71]:
tutorial_step('Task 7: 7. Filter rare features', r"""
qiime feature-table filter-features \
  --i-table asv-table.qza \
  --p-min-samples 2 \
  --o-filtered-table asv-table-ms2.qza
""")

Command to run: qiime feature-table filter-features --i-table asv-table.qza --p-min-samples 2 --o-filtered-table asv-table-ms2.qza
Already completed: Task 7: 7. Filter rare features


### Task 8: Filter sequences

Find the command beginning **`qiime feature-table filter-seqs`**.

**Expected output(s):** `asv-seqs-ms2.qza`.


In [72]:
tutorial_step('Task 8: Filter sequences', r"""
qiime feature-table filter-seqs \
  --i-data asv-seqs.qza \
  --i-table asv-table-ms2.qza \
  --o-filtered-data asv-seqs-ms2.qza
""")


Command to run: qiime feature-table filter-seqs --i-data asv-seqs.qza --i-table asv-table-ms2.qza --o-filtered-data asv-seqs-ms2.qza
Already completed: Task 8: Filter sequences


### Task 9: Filtered table summary

Find the command beginning **`qiime feature-table summarize`**.

**Expected output(s):** `asv-table-ms2.qzv`, `sample-frequencies-ms2.qza`, `asv-frequencies-ms2.qza`.


In [73]:
tutorial_step('Task 9: Filtered table summary', r"""
qiime feature-table summarize \
  --i-table asv-table-ms2.qza \
  --m-metadata-file sample-metadata.tsv \
  --o-summary asv-table-ms2.qzv \
  --o-sample-frequencies sample-frequencies-ms2.qza \
  --o-feature-frequencies asv-frequencies-ms2.qza
""")


Command to run: qiime feature-table summarize --i-table asv-table-ms2.qza --m-metadata-file sample-metadata.tsv --o-summary asv-table-ms2.qzv --o-sample-frequencies sample-frequencies-ms2.qza --o-feature-frequencies asv-frequencies-ms2.qza
Already completed: Task 9: Filtered table summary


In [74]:
view_outputs()


Dropdown(description='Result:', layout=Layout(width='90%'), options=(('alpha-rarefaction.qzv', '/content/gut_t…

Button(description='Download selected .qzv', style=ButtonStyle())

**Pause and explain:** How many ASVs were removed? Explain why excluding single-sample features might change an estimate of richness.

**Your response:**

1,069 ASVs remained inthe previous step, and now at this step, 335 (unique features, ASVS) remain. This means that 734 ASVs have since been removed. Excluding single-sample features might change an estimate of richness because it may have been a completely legitimate sample/species that just had a very low abundance. Removing it would decrease the estimate of richness.

## 8. Reference sequences and taxonomy labels

**Find in the tutorial:** Taxonomic annotation → Training a taxonomic classifier.

Before executing, identify what each input contains and predict what this step will produce.


**Provided reference download:** These are the release-matched GTDB 202.0 artifacts used in the working instructor notebook. Skip the tutorial’s `rescript get-gtdb-data` command; continue with the reference-sequence summary. The downstream model is trained by you.


In [75]:
fetch('reference-sequences.qza')
fetch('reference-taxonomy.qza')


PosixPath('/content/gut_to_soil_student/analysis/reference-taxonomy.qza')

### Task 10: Reference summary

Find the command beginning **`qiime feature-table tabulate-seqs`**.

**Expected output(s):** `reference-taxonomy.qzv`.


In [76]:
tutorial_step('Task 10: Reference summary', r"""
qiime feature-table tabulate-seqs \
  --i-data reference-sequences.qza \
  --i-taxonomy GTDB_r202_SpeciesReps:reference-taxonomy.qza \
  --p-merge-method intersect \
  --o-visualization reference-taxonomy.qzv
""")


Command to run: qiime feature-table tabulate-seqs --i-data reference-sequences.qza --i-taxonomy GTDB_r202_SpeciesReps:reference-taxonomy.qza --p-merge-method intersect --o-visualization reference-taxonomy.qzv
Already completed: Task 10: Reference summary


In [77]:
view_outputs()


Dropdown(description='Result:', layout=Layout(width='90%'), options=(('alpha-rarefaction.qzv', '/content/gut_t…

Button(description='Download selected .qzv', style=ButtonStyle())

**Pause and explain:** What are the training examples and their labels for taxonomy classification? How does the reference set limit the predictions?

**Your response:**

The training examples are different taxa of Bacteria and Archaea. Their labels for taxonomic classification include the entire nested heirarchy of classification groups separated by underscores: d__Archaea;p__Halobacteriota;c__Halobacteria;o__Halobacteriales;f__Haloferacaceae;g__Halorubrum;s__Halorubrum hochstenium. (Domain, Phyla, Class, Order, Family, Genus, Species). An example of the names these training taxa have is "RS_GCF_000337075.1". This reference set limits the predictions to organisms listed in this group. If any samples were collected that are legitimate organisms that just so happen not to match to any of the references given, the data could not be used despite it being viable.

## 9. Train a naïve Bayes taxonomy classifier

**Find in the tutorial:** Taxonomic annotation → Training a taxonomic classifier / Annotating an Artifact or Visualization.

Before executing, identify what each input contains and predict what this step will produce.


### Task 11: Train taxonomy classifier

Find the command beginning **`qiime feature-classifier fit-classifier-naive-bayes`**.

**Expected output(s):** `suboptimal-16S-rRNA-classifier.qza`.


In [78]:
tutorial_step('Task 11: Train taxonomy classifier', r"""
qiime feature-classifier fit-classifier-naive-bayes \
  --i-reference-reads reference-sequences.qza \
  --i-reference-taxonomy reference-taxonomy.qza \
  --p-feat-ext--n-features 2048 \
  --p-classify--chunk-size 2000 \
  --o-classifier suboptimal-16S-rRNA-classifier.qza
""")


Command to run: qiime feature-classifier fit-classifier-naive-bayes --i-reference-reads reference-sequences.qza --i-reference-taxonomy reference-taxonomy.qza --p-feat-ext--n-features 2048 --p-classify--chunk-size 200 --o-classifier suboptimal-16S-rRNA-classifier.qza
Already completed: Task 11: Train taxonomy classifier


### Task 12: Annotate classifier

Find the command beginning **`qiime tools annotation-create`**.

**Expected output(s):** `suboptimal-16S-rRNA-classifier.qza`.


Copy the tutorial’s complete annotation command. The adapter saves its result as `suboptimal-16S-rRNA-classifier.qza`, and uses that copy in the next task.


In [79]:
tutorial_step('Task 12: Annotate classifier', r"""
qiime tools annotation-create \
  --input-path suboptimal-16S-rRNA-classifier.qza \
  --annotation-type Note \
  --name 'suboptimal-classifier-justification' \
  --text 'This classifier was trained with non-default parameters for demonstration purposes. '\
'It should not be used in real-world analysis.' \
  --output-path suboptimal-16S-rRNA-classifier.qza""")


Command to run: qiime tools annotation-create --input-path suboptimal-16S-rRNA-classifier.qza --annotation-type Note --name suboptimal-classifier-justification --text 'This classifier was trained with non-default parameters for demonstration purposes. It should not be used in real-world analysis.' --output-path teaching-classifier.qza
Already completed: Task 12: Annotate classifier


In [80]:
view_outputs()


Dropdown(description='Result:', layout=Layout(width='90%'), options=(('alpha-rarefaction.qzv', '/content/gut_t…

Button(description='Download selected .qzv', style=ButtonStyle())

**Pause and explain:** What do k-mer features encode? Explain why changing chunk size is different from changing the number of feature buckets.

**Your response:**

k-mer features encode information about each of the ASV sequences. changing the chunk size is different from changing the number of feature buckets because chunk size determines how many ASV sequences you look at at once and the number of feature buckets determines how many groups are made of the kmers being considered. only the number of feature buckets impacts how the data is representated.


## 10. Assign taxonomy

**Find in the tutorial:** Apply our taxonomy classifier.

Before executing, identify what each input contains and predict what this step will produce.


### Task 13: 10. Assign taxonomy

Find the command beginning **`qiime feature-classifier classify-sklearn`**.

**Expected output(s):** `taxonomy.qza`.


In [81]:
tutorial_step('Task 13: 10. Assign taxonomy', r"""
qiime feature-classifier classify-sklearn \
  --i-classifier suboptimal-16S-rRNA-classifier.qza \
  --i-reads asv-seqs-ms2.qza \
  --o-classification taxonomy.qza
""")


Command to run: qiime feature-classifier classify-sklearn --i-classifier teaching-classifier.qza --i-reads asv-seqs-ms2.qza --o-classification taxonomy.qza --p-n-jobs 1 --p-reads-per-batch 100
Already completed: Task 13: 10. Assign taxonomy


### Task 14: Annotated ASVs

Find the command beginning **`qiime feature-table tabulate-seqs`**.

**Expected output(s):** `asv-seqs-ms2.qzv`.


In [82]:
tutorial_step('Task 14: Annotated ASVs', r"""
qiime feature-table tabulate-seqs \
  --i-data asv-seqs-ms2.qza \
  --i-taxonomy GTDB_r202_SpeciesReps:taxonomy.qza \
  --m-metadata-file asv-frequencies-ms2.qza \
  --o-visualization asv-seqs-ms2.qzv
""")


Command to run: qiime feature-table tabulate-seqs --i-data asv-seqs-ms2.qza --i-taxonomy GTDB_r202_SpeciesReps:taxonomy.qza --m-metadata-file asv-frequencies-ms2.qza --o-visualization asv-seqs-ms2.qzv
Already completed: Task 14: Annotated ASVs


In [83]:
view_outputs()


Dropdown(description='Result:', layout=Layout(width='90%'), options=(('alpha-rarefaction.qzv', '/content/gut_t…

Button(description='Download selected .qzv', style=ButtonStyle())

**Pause and explain:** Choose one ASV. Record its taxonomy and deepest assigned rank. Why might an assignment stop above species level?

**Your response:**

An assignment might stop above species level if the ASV only matches to the reference to a certain level (ex. family or genus) and does not contain the correct nucleotides to be considered a match to any of the species references. (ex. if the ASV sequence does not match any of the given reference sequences down to the species level, but matches the reference until the genus level instead.)

ASV: 00a96fbd0ac34bfd245f9c24f8737f7d

Taxonomy: d__Bacteria;p__Firmicutes_A;c__Clostridia;o__Lachnospirales;f__Lachnospiraceae;g__Blautia_A;s__Blautia_A sp000436615

Deepest assigned rank: species level, blautia




## 11. Community composition

**Find in the tutorial:** Taxonomic analysis.

Before executing, identify what each input contains and predict what this step will produce.


### Task 15: 11. Community composition

Find the command beginning **`qiime taxa barplot`**.

**Expected output(s):** `taxa-bar-plots.qzv`.


In [84]:
tutorial_step('Task 15: 11. Community composition', r"""
qiime taxa barplot \
  --i-table asv-table-ms2.qza \
  --i-taxonomy taxonomy.qza \
  --m-metadata-file sample-metadata.tsv \
  --o-visualization taxa-bar-plots.qzv
""")


Command to run: qiime taxa barplot --i-table asv-table-ms2.qza --i-taxonomy taxonomy.qza --m-metadata-file sample-metadata.tsv --o-visualization taxa-bar-plots.qzv
Already completed: Task 15: 11. Community composition


In [85]:
view_outputs()


Dropdown(description='Result:', layout=Layout(width='90%'), options=(('alpha-rarefaction.qzv', '/content/gut_t…

Button(description='Download selected .qzv', style=ButtonStyle())

**Pause and explain:** Compare two focal sample types at two taxonomic levels. Explain why relative abundance does not establish absolute abundance.

**Your response:**

at taxonomic level 1, the sample b0642 is predominately bacterial species, with ~10% species of archaea detected. at taxonomic level 1, the sample 016287d9 is comprised of 100%- bacterial species. at taxonomic level 2, it can be observed that sample b0642 hosted bacteria from 2 of the same phyla as 016287d9: p__Campylobacterota and p__Desulfobacterota. Relative abundance does not establish absolute abundance because it focuses on the amount/prevalance of specific organisms in a sample and is not characterisic of everything that may be encountered there. focusing only on relative abundance may give a warped perception of community competition.


## 12. Rarefaction

**Find in the tutorial:** Selecting an even sampling depth.

Before executing, identify what each input contains and predict what this step will produce.


### Task 16: 12. Rarefaction

Find the command beginning **`qiime diversity alpha-rarefaction`**.

**Expected output(s):** `alpha-rarefaction.qzv`.


In [86]:
tutorial_step('Task 16: 12. Rarefaction', r"""
qiime diversity alpha-rarefaction \
  --i-table asv-table-ms2.qza \
  --p-max-depth 260 \
  --m-metadata-file sample-metadata.tsv \
  --o-visualization alpha-rarefaction.qzv
""")


Command to run: qiime diversity alpha-rarefaction --i-table asv-table-ms2.qza --p-max-depth 260 --m-metadata-file sample-metadata.tsv --o-visualization alpha-rarefaction.qzv
Already completed: Task 16: 12. Rarefaction


In [87]:
view_outputs()


Dropdown(description='Result:', layout=Layout(width='90%'), options=(('alpha-rarefaction.qzv', '/content/gut_t…

Button(description='Download selected .qzv', style=ButtonStyle())

**Pause and explain:** Compare sampling depths using both diversity curves and sample retention. Justify a depth, including its tradeoff.

**Your response:**

The sequencing depth for several samples plateaus between 40-80. When observing the alpha rarefaction figures, it appears that for human excrement samples, the best sequencing depth is ~60, because no other sequencing depths provide much a much higher shannon index than the score of ~4.0 at a seq depth of 60. however, when comparing this to the diversity curves, a sequencing depth of 140 appears to reveal more of the diversity of the sample (22-24 observed features) than at sequencing depth of 60 (20 OF). In this case, the best depth may be 140 because even though it is a few steps into the initial plateau, it retains more species diversity than a depth of 60 and maintains the 4.0 score on the shannon index. For some other samples, a higher sequencing depth provides a higher shannon score, but retains fewer observed species on the diversity curve (tradeoff).

## 13. Alpha diversity, beta diversity, and ordination

**Find in the tutorial:** Running the kmer-diversity Pipeline.

Before executing, identify what each input contains and predict what this step will produce.


### Task 17: Kmer diversity

Find the command beginning **`qiime boots kmer-diversity`**.

**Expected output(s):** `boots-kmer-diversity`.


In [88]:
tutorial_step('Task 17: Kmer diversity', r"""
qiime boots kmer-diversity \
  --i-table asv-table-ms2.qza \
  --i-sequences asv-seqs-ms2.qza \
  --m-metadata-file sample-metadata.tsv \
  --p-sampling-depth 96 \
  --p-n 10 \
  --p-no-replacement \
  --output-dir boots-kmer-diversity
""")


Command to run: qiime boots kmer-diversity --i-table asv-table-ms2.qza --i-sequences asv-seqs-ms2.qza --m-metadata-file sample-metadata.tsv --p-sampling-depth 96 --p-n 10 --p-no-replacement --output-dir boots-kmer-diversity
Already completed: Task 17: Kmer diversity


In [89]:
view_outputs()


Dropdown(description='Result:', layout=Layout(width='90%'), options=(('alpha-rarefaction.qzv', '/content/gut_t…

Button(description='Download selected .qzv', style=ButtonStyle())

**Pause and explain:** Which categories separate in PCoA? Explain how PCoA differs from PCA. Does separation alone demonstrate predictive accuracy?

**Your response:**

In the PCoA, samples from human excrement/human excrement compost separate together from food compost and soil samples.PCoA differs from PCA in that PCoA reveals dissimilarity between samples, whereas PCA reveals the relationship each individual sample shares with the varaible being considered (variance among features). Separation alone does not demonstrate predictive accuracy, since all it is revealing is the relationship between the samples.

## 14. Restrict the differential-abundance comparison

**Find in the tutorial:** Differential abundance testing with ANCOM-BC2.

Before executing, identify what each input contains and predict what this step will produce.


### Task 18: 14. Restrict the differential-abundance comparison

Find the command beginning **`qiime feature-table filter-samples`**.

**Expected output(s):** `asv-table-ms2-dominant-sample-types.qza`.


In [90]:
tutorial_step('Task 18: 14. Restrict the differential-abundance comparison', r"""
qiime feature-table filter-samples \
  --i-table asv-table-ms2.qza \
  --m-metadata-file sample-metadata.tsv \
  --p-where '[SampleType] IN ("Human Excrement Compost", "Human Excrement", "Food Compost")' \
  --o-filtered-table asv-table-ms2-dominant-sample-types.qza
""")


Command to run: qiime feature-table filter-samples --i-table asv-table-ms2.qza --m-metadata-file sample-metadata.tsv --p-where '[SampleType] IN ("Human Excrement Compost", "Human Excrement", "Food Compost")' --o-filtered-table asv-table-ms2-dominant-sample-types.qza
Already completed: Task 18: 14. Restrict the differential-abundance comparison


In [91]:
tutorial_step("summarize-dominant-sample-types", r"""
qiime feature-table summarize \
  --i-table asv-table-ms2-dominant-sample-types.qza \
  --m-metadata-file sample-metadata.tsv \
  --o-summary asv-table-ms2-dominant-sample-types.qzv \
  --o-sample-frequencies dominant-sample-frequencies.qza \
  --o-feature-frequencies dominant-feature-frequencies.qza
""")

Command to run: qiime feature-table summarize --i-table asv-table-ms2-dominant-sample-types.qza --m-metadata-file sample-metadata.tsv --o-summary asv-table-ms2-dominant-sample-types.qzv --o-sample-frequencies dominant-sample-frequencies.qza --o-feature-frequencies dominant-feature-frequencies.qza
Already completed: summarize-dominant-sample-types


In [92]:
view_outputs()


Dropdown(description='Result:', layout=Layout(width='90%'), options=(('alpha-rarefaction.qzv', '/content/gut_t…

Button(description='Download selected .qzv', style=ButtonStyle())

**Pause and explain:** Which three sample types are retained? Why is this count table not rarefied before ANCOM-BC2?

**Your response:**

The three sample types retained are human excrement, human excrement compose, and food compost. This count table is not rarefied before ANCOM-BC2 because rarefying would remove data that could otherwise be viable for downstream analyses. instead, performing ANCOM-BC2 on thr count table will correct for sequencing depth without removing viable sequences that have different depths.


## 15. ANCOM-BC2 at ASV level

**Find in the tutorial:** Differential abundance testing with ANCOM-BC2.

Before executing, identify what each input contains and predict what this step will produce.


### Task 19: 15. ANCOM-BC2 at ASV level

Find the command beginning **`qiime composition ancombc2`**.

**Expected output(s):** `ancombc2-results.qza`.


In [93]:
tutorial_step('Task 19: 15. ANCOM-BC2 at ASV level', r"""
qiime composition ancombc2 \
  --i-table asv-table-ms2-dominant-sample-types.qza \
  --m-metadata-file sample-metadata.tsv \
  --p-fixed-effects-formula SampleType \
  --p-reference-levels 'SampleType::Human Excrement Compost' \
  --o-ancombc2-output ancombc2-results.qza
""")


Command to run: qiime composition ancombc2 --i-table asv-table-ms2-dominant-sample-types.qza --m-metadata-file sample-metadata.tsv --p-fixed-effects-formula SampleType --p-reference-levels 'SampleType::Human Excrement Compost' --o-ancombc2-output ancombc2-results.qza
Already completed: Task 19: 15. ANCOM-BC2 at ASV level


### Task 20: ASV differential-abundance plot

Find the command beginning **`qiime composition da-barplot`**.

**Expected output(s):** `ancombc2-barplot.qzv`.


In [94]:
tutorial_step('Task 20: ASV differential-abundance plot', r"""
qiime composition da-barplot \
  --i-data ancombc2-results.qza \
  --i-taxonomy taxonomy.qza \
  --o-visualization ancombc2-barplot.qzv
""")


Command to run: qiime composition da-barplot --i-data ancombc2-results.qza --i-taxonomy taxonomy.qza --o-visualization ancombc2-barplot.qzv
Already completed: Task 20: ASV differential-abundance plot


In [95]:
view_outputs()


Dropdown(description='Result:', layout=Layout(width='90%'), options=(('alpha-rarefaction.qzv', '/content/gut_t…

Button(description='Download selected .qzv', style=ButtonStyle())

**Pause and explain:** Choose one ASV comparison. Report direction, effect size, and adjusted evidence. What does the reference level mean?

**Your response:**

ASV: (c6c3ab) g__Blautia_A

Direction: positive

Effect size: ~1.9-2.4 (including farthest error bar)

Adjusted evidence: p value < 0.05

The reference level is what all of the samples are being compared to. For this figure, the reference level is human excrement compost, and all of the log values associated with the other samples is being compared to this baseline.

## 16. Repeat at genus level

**Find in the tutorial:** Exercise 15: Perform ANCOM-BC2 on genera, instead of ASVs.

Before executing, identify what each input contains and predict what this step will produce.


### Task 21: 16. Repeat at genus level

Find the command beginning **`qiime taxa collapse`**.

**Expected output(s):** `genus-table-ms2-dominant-sample-types.qza`.


In [96]:
tutorial_step('Task 21: 16. Repeat at genus level', r"""
qiime taxa collapse \
  --i-table asv-table-ms2-dominant-sample-types.qza \
  --i-taxonomy taxonomy.qza \
  --p-level 6 \
  --o-collapsed-table genus-table-ms2-dominant-sample-types.qza
""")


Command to run: qiime taxa collapse --i-table asv-table-ms2-dominant-sample-types.qza --i-taxonomy taxonomy.qza --p-level 6 --o-collapsed-table genus-table-ms2-dominant-sample-types.qza
Already completed: Task 21: 16. Repeat at genus level


### Task 22: Genus ANCOM-BC2

Find the command beginning **`qiime composition ancombc2`**.

**Expected output(s):** `genus-ancombc2-results.qza`.


In [97]:
tutorial_step('Task 22: Genus ANCOM-BC2', r"""
qiime composition ancombc2 \
  --i-table genus-table-ms2-dominant-sample-types.qza \
  --m-metadata-file sample-metadata.tsv \
  --p-fixed-effects-formula SampleType \
  --p-reference-levels 'SampleType::Human Excrement Compost' \
  --o-ancombc2-output genus-ancombc2-results.qza
""")


Command to run: qiime composition ancombc2 --i-table genus-table-ms2-dominant-sample-types.qza --m-metadata-file sample-metadata.tsv --p-fixed-effects-formula SampleType --p-reference-levels 'SampleType::Human Excrement Compost' --o-ancombc2-output genus-ancombc2-results.qza
Already completed: Task 22: Genus ANCOM-BC2


### Task 23: Genus differential-abundance plot

Find the command beginning **`qiime composition da-barplot`**.

**Expected output(s):** `genus-ancombc2-barplot.qzv`.


In [98]:
tutorial_step('Task 23: Genus differential-abundance plot', r"""
qiime composition da-barplot \
  --i-data genus-ancombc2-results.qza \
  --o-visualization genus-ancombc2-barplot.qzv
""")


Command to run: qiime composition da-barplot --i-data genus-ancombc2-results.qza --o-visualization genus-ancombc2-barplot.qzv
Already completed: Task 23: Genus differential-abundance plot


In [99]:
view_outputs()


Dropdown(description='Result:', layout=Layout(width='90%'), options=(('alpha-rarefaction.qzv', '/content/gut_t…

Button(description='Download selected .qzv', style=ButtonStyle())

**Pause and explain:** Did aggregation to genus clarify or obscure a result? Explain why results can differ at ASV and genus levels.

**Your response:**

Although the result I observed at the ASV level is still supported at the genus level, it is less clear what the specific outcome for the selected sample is when it is grouped with the rest of the ASVs of the genus. Results may differ at ASV and genus levels because ASV-specific results are no longer individually represented by the plot, and are instead grouped together at the genus level. Grouping at this level can change the direction or magnitude of the result because it is more of a broader box containing multiple fine-scale results. (ie. the genus level contains multiple differnt fine scale ASV results combined, and this combination may reduce the effect size observed in an individual ASV)

## 17. Reconstruct provenance

**Find in the tutorial:** Ensuring bioinformatics reproducibility and adapting the tutorial workflow for your own data.

Before executing, identify what each input contains and predict what this step will produce.


### Task 24: Replay provenance

Paste only the `qiime tools replay-provenance` command. Skip `cd`: the adapter supplies the correct folders. Expected output: `g2s-replayed.bash`.


In [100]:
tutorial_step('Task 24: Replay provenance', r"""
qiime tools replay-provenance \
  --out-fp g2s-replayed.bash
""")


$ /content/gut_to_soil_student/bin/micromamba run -p /content/gut_to_soil_student/qiime-2026.10 qiime tools replay-provenance --out-fp /content/gut_to_soil_student/g2s-replayed.bash --in-fp /content/gut_to_soil_student/analysis
parsing /content/gut_to_soil_student/analysis/genus-table-ms2-dominant-sample-types.qza
parsing /content/gut_to_soil_student/analysis/teaching-classifier.qza
parsing /content/gut_to_soil_student/analysis/taxonomy.qza
parsing /content/gut_to_soil_student/analysis/denoising-stats.qza
parsing /content/gut_to_soil_student/analysis/genus-ancombc2-barplot.qzv
parsing /content/gut_to_soil_student/analysis/base-transition-stats.qza
parsing /content/gut_to_soil_student/analysis/genus-ancombc2-results.qza
parsing /content/gut_to_soil_student/analysis/asv-seqs.qza
parsing /content/gut_to_soil_student/analysis/reference-sequences.qza
parsing /content/gut_to_soil_student/analysis/ancombc2-results.qza
parsing /content/gut_to_soil_student/analysis/asv-table-ms2.qzv
parsing /co

**Pause and explain:** Find the DADA2 trimming settings and diversity sampling depth in the replay script. Explain why it may contain commands you did not run yourself.

**Your response:**

DADA2 trimming settings: trim left f 0
trunc len f 250
trim left r 0
trunc len r 250

diversity sampling depth, max depth: 96

As mentioned in the tutorial page, for reproducability, these steps are contained in data provenance by the creators of the tutorial. this documents the steps run to acquire each file in the tutorial and ensures that the same results can be achieved each time someone works through the cells.

## Exit Questions

### 1. What do the microbial communities suggest?

Do human-excrement compost communities resemble human excrement, food compost, or fall between them? Support your interpretation with two specific results or visualizations from your notebook. Describe any variation or exceptions that affect your conclusion.

**Your response:** I found that human excrement compost communities resemble food compost more closely, though to varying degrees. When considering the scatterplot, the data points representing samples from human excrement grouped completely separately from human excrement compost and all other sample types (human excrement compost grouped with food compost and other sample types).

### 2. Could a classifier detect a fecal microbial signature?

Suppose you trained a classifier to distinguish human excrement from food compost, then applied it to human-excrement compost samples. What would its predictions tell you? Why would an “excrement-like” prediction not establish contamination or show that the compost is unsafe?

**Your response:** I would guess that the predictions would tell me my compost samples are from food compost instead of from human excrement. An excrement-like prediction would not establish contamination or show that the compost is unsafe because the classifier would only be able to determine whether the samples i give it have a microbial signature more similar to human excrement or food compost, not whether or not the compost is unsafe or if it has been contaminated.  

### 3. How should we evaluate predictions?

Several samples came from the same composting bucket. Why could randomly dividing samples into training and test sets overestimate performance on new buckets? Describe a better way to split the data and explain what that evaluation would test.

**Your response:** Randomly dividing samples from the same composting bucket and using the random groups as training sets could overestimate performance on new buckets because the training and validation data would be coming from the same place - there is effectively no "new" data that was withheld. a radnom sample from the same bucket would represent the random variation of the entire bucket, so introducting a different random sample from the same bucket would not be truly representative of how well the classifier would do on new, unseen data. a better way to split the data would be to use samples from the compost bucket within a certain time frame for training, then use samples from the bucket from a different time frame for validation. this would test/examine how the microbial composition of that bucket may have changed over time and would effectively simulate 2 different datasets for training vs validation.

## Optional course extension — Sample-level machine learning
The official tutorial ends with provenance. The following **provided** extension connects your ASV table to the ML material from class. Run it after completing the tutorial, if directed. Read the explanation before running; predictions do not require the differential-abundance results.

**Predict first:** Which pair of sample types will be hardest to distinguish, and why?

**Your prediction:** [Write here.]


### Read before running the provided ML extension

This is an added course exercise, separate from the core tutorial. It runs in QIIME's environment so Colab does not need additional scientific packages installed.

**Features:** ASV counts from the unfiltered table. **Target:** SampleType for the three focal categories. The workflow makes a stratified 75/25 split, learns a prevalence filter on training data, uses a Hellinger transform (square roots of per-sample relative abundances), and fits a random forest. A majority-class baseline and shuffled-training-label control contextualize performance. The held-out set is for one evaluation, not repeated hyperparameter selection.

For ordination, PCA is fitted on transformed training data, then test samples are projected. PCA is for visualization here and is not an input to the classifier. ASV inference itself used all reads, as in the tutorial; this is an instructional evaluation within one sequencing study, not an independent external validation.

The script also reports bucket overlap and attempts grouped cross-validation. Missing bucket identifiers receive sample-specific groups, which **does not establish independence**. Shared donors, sites, or collection batches may require stronger grouping. Review the metadata before making claims about generalization.


In [ ]:
ml_script = ROOT / 'ml_extension.py'
ml_script.write_text('"""Teaching extension. Use --synthetic-test for a mechanics-only smoke test."""\nfrom pathlib import Path\nimport sys, json\nimport numpy as np\nimport pandas as pd\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nfrom sklearn.base import BaseEstimator, TransformerMixin\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.dummy import DummyClassifier\nfrom sklearn.decomposition import PCA\nfrom sklearn.model_selection import train_test_split, StratifiedGroupKFold, cross_validate\nfrom sklearn.metrics import (accuracy_score, balanced_accuracy_score, classification_report,\n                             ConfusionMatrixDisplay)\n\n\nclass PrevalenceHellinger(BaseEstimator, TransformerMixin):\n    def __init__(self, min_samples=2):\n        self.min_samples=min_samples\n\n    def fit(self, X, y=None):\n        X=np.asarray(X, dtype=float)\n        self.keep_=(X>0).sum(axis=0)>=self.min_samples\n        if not self.keep_.any(): raise ValueError(\'No features survive training-only prevalence filtering\')\n        return self\n\n    def transform(self, X):\n        X=np.asarray(X,dtype=float)[:,self.keep_]\n        sums=X.sum(axis=1,keepdims=True)\n        # Empty rows map to zeros; report these in the main evaluation.\n        return np.sqrt(np.divide(X,sums,out=np.zeros_like(X),where=sums>0))\n\n\ndef main(work, seed=42, synthetic=False):\n    out=work/\'ml\'; out.mkdir(parents=True,exist_ok=True)\n    labels=[\'Food Compost\',\'Human Excrement\',\'Human Excrement Compost\']\n    if synthetic:\n        rng=np.random.default_rng(seed)\n        y0=np.repeat(labels,20)\n        raw=rng.poisson(2,(60,30))\n        for k in range(3): raw[k*20:(k+1)*20,k*10:(k+1)*10]+=rng.poisson(4,(20,10))\n        X=pd.DataFrame(raw,index=[f\'s{i}\' for i in range(60)],columns=[f\'asv{i}\' for i in range(30)])\n        metadata=pd.DataFrame({\'SampleType\':y0,\'Bucket\':[str(i//3) for i in range(60)]},index=X.index)\n    else:\n        from biom import load_table\n        import zipfile, tempfile\n        with tempfile.TemporaryDirectory() as temp:\n            with zipfile.ZipFile(work/\'asv-table.qza\') as z:\n                names=[n for n in z.namelist() if n.endswith(\'/data/feature-table.biom\')]\n                if len(names)!=1: raise ValueError(\'Cannot locate feature table inside QIIME artifact\')\n                data=Path(temp)/\'feature-table.biom\'\n                with z.open(names[0]) as src, data.open(\'wb\') as dst:\n                    import shutil\n                    shutil.copyfileobj(src,dst)\n            table=load_table(str(data))\n            # BIOM stores features in rows; ML requires samples in rows.\n            X=table.to_dataframe(dense=True).T\n        metadata=pd.read_csv(work/\'sample-metadata.tsv\',sep=\'\\t\',index_col=0,dtype=str)\n        metadata=metadata.loc[~metadata.index.astype(str).str.startswith(\'#\')]\n        if not metadata.index.is_unique: raise ValueError(\'Duplicate sample identifiers in metadata\')\n        missing=X.index.difference(metadata.index)\n        if len(missing): raise ValueError(f\'Sample IDs missing metadata: {list(missing)}\')\n        metadata=metadata.reindex(X.index)\n    selected=metadata[\'SampleType\'].isin(labels) & (X.sum(axis=1)>0)\n    X=X.loc[selected].astype(float); metadata=metadata.loc[X.index]\n    y=metadata[\'SampleType\']\n    if y.nunique()!=3 or y.value_counts().min()<4:\n        raise ValueError(\'Insufficient samples per class for this teaching split\')\n    train,test=train_test_split(np.arange(len(X)),test_size=.25,stratify=y,random_state=seed)\n    Xtrain,Xtest=X.iloc[train],X.iloc[test]\n    ytrain,ytest=y.iloc[train],y.iloc[test]\n    split=pd.DataFrame({\'SampleType\':y,\'partition\':\'train\'})\n    split.loc[Xtest.index,\'partition\']=\'test\'; split.to_csv(out/\'split.tsv\',sep=\'\\t\')\n    X.to_csv(out/\'asv-counts.tsv\',sep=\'\\t\'); metadata.to_csv(out/\'metadata.tsv\',sep=\'\\t\')\n    def forest():\n        return make_pipeline(PrevalenceHellinger(2),RandomForestClassifier(\n            n_estimators=200,min_samples_leaf=2,class_weight=\'balanced\',random_state=seed,n_jobs=1))\n    models={\'Majority baseline\':DummyClassifier(strategy=\'most_frequent\'),\n            \'Random forest\':forest(),\'Shuffled training labels\':forest()}\n    lines=[\'SYNTHETIC MECHANICS TEST — NOT BIOLOGICAL RESULTS\' if synthetic else \'Gut-to-soil sample classification\',\n           f\'Samples: {X.shape[0]}; ASVs: {X.shape[1]}; train: {len(train)}; test: {len(test)}\',\n           \'Class counts:\\n\'+y.value_counts().to_string()]\n    fig,axes=plt.subplots(1,3,figsize=(16,5))\n    metrics=[]\n    for ax,(name,model) in zip(axes,models.items()):\n        target=np.random.default_rng(seed).permutation(ytrain.to_numpy()) if name.startswith(\'Shuffled\') else ytrain\n        model.fit(Xtrain,target); pred=model.predict(Xtest)\n        score=dict(model=name,accuracy=accuracy_score(ytest,pred),balanced_accuracy=balanced_accuracy_score(ytest,pred))\n        metrics.append(score)\n        lines.append(\'\\n\'+name+\'\\n\'+classification_report(ytest,pred,labels=labels,zero_division=0))\n        ConfusionMatrixDisplay.from_predictions(ytest,pred,labels=labels,\n             display_labels=[\'Food\',\'Excrement\',\'HE compost\'],ax=ax,colorbar=False,cmap=\'Blues\')\n        ax.set_title(name,fontsize=11)\n    fig.tight_layout(); fig.savefig(out/\'confusion-matrices.png\',dpi=160); plt.close(fig)\n    pd.DataFrame(metrics).to_csv(out/\'metrics.tsv\',sep=\'\\t\',index=False)\n    fitted=models[\'Random forest\']; transform=fitted.steps[0][1]\n    Htrain=transform.transform(Xtrain); Htest=transform.transform(Xtest)\n    lines.append(f\'ASVs retained using training data: {transform.keep_.sum()}\')\n    lines.append(f\'Test samples with no retained counts: {int((Htest.sum(axis=1)==0).sum())}\')\n    if min(Htrain.shape)>=2:\n        pca=PCA(n_components=2).fit(Htrain)\n        fig,ax=plt.subplots(figsize=(8,6))\n        for part,data,target,marker in [(\'train\',Htrain,ytrain,\'o\'),(\'test\',Htest,ytest,\'^\')]:\n            coords=pca.transform(data)\n            for label,color in zip(labels,[\'#238b45\',\'#9e3d78\',\'#cc7a00\']):\n                mask=target.to_numpy()==label\n                ax.scatter(coords[mask,0],coords[mask,1],c=color,marker=marker,alpha=.8,label=f\'{label} ({part})\')\n        ax.set(xlabel=f\'PC1 ({pca.explained_variance_ratio_[0]:.1%})\',ylabel=f\'PC2 ({pca.explained_variance_ratio_[1]:.1%})\',\n               title=\'PCA of Hellinger-transformed ASVs\\nTransform and PCA fitted on training samples\')\n        ax.legend(fontsize=8); fig.tight_layout(); fig.savefig(out/\'pca.png\',dpi=160); plt.close(fig)\n    importance=pd.Series(fitted.steps[-1][1].feature_importances_,index=X.columns[transform.keep_]).sort_values(ascending=False)\n    importance.rename(\'impurity_importance\').to_csv(out/\'feature-importance.tsv\',sep=\'\\t\')\n    fig,ax=plt.subplots(figsize=(10,6))\n    importance.head(12).sort_values().plot.barh(ax=ax,color=\'#2a6f97\')\n    ax.set(xlabel=\'Random forest impurity importance (training fit)\',title=\'Predictive features: association, not causation\')\n    ax.tick_params(axis=\'y\',labelsize=7); fig.tight_layout(); fig.savefig(out/\'feature-importance.png\',dpi=160); plt.close(fig)\n    lines.append(\'Feature importance is training-derived impurity importance; correlated features can share importance.\')\n    if \'Bucket\' in metadata:\n        buckets=metadata[\'Bucket\'].fillna(\'\').str.strip()\n        missing=buckets.str.lower().isin([\'\',\'nan\',\'na\',\'n/a\',\'none\',\'not applicable\',\'not provided\',\'missing\'])\n        groups=pd.Series([\'sample:\'+str(i) if absent else \'bucket:\'+str(b)\n                          for i,b,absent in zip(X.index,buckets,missing)],index=X.index)\n        overlap=set(groups.iloc[train]) & set(groups.iloc[test])\n        lines.append(f\'Groups overlapping random training/test partitions: {sorted(overlap)}\')\n        lines.append(f\'Samples lacking bucket IDs: {int(missing.sum())}; independence of their sample-specific groups is unverified.\')\n        if groups.nunique()>=3:\n            cv=StratifiedGroupKFold(n_splits=3,shuffle=True,random_state=seed)\n            folds=list(cv.split(X,y,groups))\n            usable=all(set(y.iloc[a])==set(labels) and set(y.iloc[b])==set(labels) for a,b in folds)\n            if usable:\n                scores=cross_validate(forest(),X,y,cv=folds,scoring=\'balanced_accuracy\',error_score=\'raise\')\n                grouped=scores[\'test_score\']\n                lines.append(\'Separate 3-fold bucket-grouped CV balanced accuracies: \'+np.array2string(grouped,precision=3))\n                lines.append(f\'Mean: {grouped.mean():.3f}. This uses different folds/data from the single holdout; it is not a paired accuracy comparison.\')\n            else:\n                lines.append(\'Grouped CV skipped: a fold lacks a focal class. Review the study design and grouping before changing the split.\')\n        lines.append(\'Bucket grouping does not guarantee donor/site independence. Review shared donors and collection sources.\')\n    else:\n        lines.append(\'No Bucket field: no grouped evaluation. Inspect metadata before treating observations as independent.\')\n    lines.append(\'This is a small within-study teaching evaluation, not evidence of deployment-ready prediction.\')\n    text=\'\\n\'.join(lines); (out/\'evaluation.txt\').write_text(text); print(text)\n\n\nif __name__==\'__main__\':\n    if \'--synthetic-test\' in sys.argv:\n        main(Path(sys.argv[1]),synthetic=True)\n    else:\n        main(Path(sys.argv[1]),int(sys.argv[2]) if len(sys.argv)>2 else 42)\n')
run([MM, 'run', '-p', PREFIX, 'python', ml_script, str(WORK), str(SEED)], 'ml-extension')


In [ ]:
from IPython.display import display, Image
for name in ['pca.png','confusion-matrices.png','feature-importance.png']:
    path=WORK/'ml'/name
    if path.exists(): display(Image(filename=str(path)))
print((WORK/'ml'/'evaluation.txt').read_text())


### Student interpretation

1. Contrast sequence-level taxonomy classification with sample-level environment classification: observations, features, labels, and reference information.
2. Compare the random forest with the baseline and shuffled-label control. Which errors dominate? Why can accuracy alone be misleading?
3. Does the random split put samples from the same bucket on both sides? How does that affect the claim being evaluated?
4. Compare PCA and PCoA. Does visible separation guarantee accurate prediction on a new biological replicate?
5. Identify one influential ASV. Does importance establish causation? Compare predictive importance with differential abundance, explaining why they need not agree.
6. State two limitations: the small, shallow dataset; compositional features; study dependence; incomplete reference taxonomy; or another evidence-based concern.

**Optional experiment:** change one model setting only after stating a prediction. For genuine model selection, use cross-validation within the training set and reserve a new final test set. Repeatedly optimizing against the displayed test score contaminates that evaluation.

**Your responses:**

[Write here.]


## Export results before leaving Colab

This archive includes analysis artifacts, plots, scripts, logs, file checksums, and environment information. It excludes the installed environment, the large input read artifact, and reference/trained-classifier artifacts, which can be regenerated or downloaded again. Each downstream QIIME artifact retains its own provenance.


In [101]:
archive=ROOT/'gut_to_soil_results.zip'
excluded={'demux.qza','reference-sequences.qza','reference-taxonomy.qza',
          'suboptimal-16S-rRNA-classifier.qza','teaching-classifier.qza'}
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for directory in [WORK,LOGS,ROOT/'reference-recovery']:
        if directory.exists():
            for p in directory.rglob('*'):
                if p.is_file() and p.name not in excluded:
                    z.write(p,p.relative_to(ROOT))
    for pattern in ['*.json','*.yml','*.py','*.bash']:
        for p in ROOT.glob(pattern): z.write(p,p.name)
print('Archive:',archive,round(archive.stat().st_size/1e6,1),'MB')
from google.colab import files
files.download(str(archive))


Archive: /content/gut_to_soil_student/gut_to_soil_results.zip 50.6 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Recovery: published visualizations without a working QIIME installation

Run section 1, then this cell if installation or a later computation prevents class from continuing. It downloads **published reference visualizations into a separate folder**, never into the computed analysis. These are not your computed results. Open them in QIIME 2 View. This recovery route supports interpretation, not a claim that the workflow ran successfully. It still requires working internet downloads.


In [ ]:
RECOVERY = False  # change to True only when intentionally using published results
if RECOVERY:
    recovery=ROOT/'reference-recovery'; recovery.mkdir(exist_ok=True)
    names=['demux.qzv','denoising-stats.qzv','asv-table-ms2.qzv','taxa-bar-plots.qzv',
           'alpha-rarefaction.qzv','boots-kmer-diversity/scatter_plot.qzv',
           'ancombc2-barplot.qzv','genus-ancombc2-barplot.qzv']
    failures=[]
    for name in names:
        try: download(BASE+name,recovery/name)
        except Exception as e: failures.append((name,str(e)))
    record('checkpoints.json',dict(step='Reference visualizations only',files=names,failures=failures))
    recovered=ROOT/'published_reference_visualizations.zip'
    with zipfile.ZipFile(recovered,'w') as z:
        for p in recovery.rglob('*.qzv'): z.write(p,p.relative_to(recovery))
    print('PUBLISHED REFERENCE RESULTS — not computed in this runtime. Failures:',failures)
    from google.colab import files
    files.download(str(recovered))
else:
    print('Recovery disabled. Your computed results are unchanged.')


## Final reflection and submission checklist

- [x] I completed the tutorial commands in order and saved the notebook with outputs.
- [x] I inspected the quality, taxonomy, ordination, and differential-abundance visualizations.
- [x] I answered the interpretation prompts using my own results.
- [x] I identified any downloaded reference results used instead of computed outputs.
- [x] I downloaded the results ZIP and saved my completed notebook.

**Synthesis (one paragraph):** What evidence supports a difference among microbial communities in the three focal environments? Distinguish descriptive patterns, statistical association, and prediction. State one limitation of this dataset or analysis.

Evidence that supports a difference among microbial communities in three focal environments are the scatterplot and the interactive bar plot. the scatter plot shows that there is a strong grouping pattern that separates human fecal samples from other sample types, and the interactive bar plot shows that the microbial distribution/composition changes significantly when comparing food compost to human excrement compost. the scatterplot represents descriptive patterns (grouping patterns in sample types) and the interactive bar plot demonstrates a statistical association (p-values associated with the log scale numbers) between microbial communities and the sample type. One limitation of this dataset is that the sequencing depth is relatively low (96) when there were sequences with much greater depths (>100). cutting it at 96 may have removed other legitimate variation that could have been considered.

### If something fails
A placeholder message means you have not pasted the command yet. An unknown option usually indicates the wrong tutorial release or an incomplete paste. Check that you used the Command Line tab, included all output flags, and retained filenames. A failed command prints its log location; preserve the error for your instructor. Do not run downstream commands until their inputs exist. Use the separately labeled recovery plots only if directed.

### Sources
[Gut-to-soil tutorial, QIIME 2 2026.10](https://gut-to-soil-tutorial.readthedocs.io/en/2026.10/), by Caporaso and colleagues, based on the study by Meilander and colleagues. The linked tutorial contains the full explanations and citations. This course scaffold preserves the working instructor notebook’s Colab setup and adds spaces for commands and interpretation. The optional sample-classification extension is course material, separate from the official tutorial.
